# Dark Manifold V29.5: Microsecond Resolution Simulation

**Key insight**: If we can simulate 20 virtual minutes in 2 real seconds,
why not use 5 real minutes to get 150x finer resolution?

## Resolution Comparison
| Version | Real Time | Resolution | Steps | Biological Capture |
|---------|-----------|------------|-------|-------------------|
| V29.4 | 2 sec | 1 ms | 1.2M | Slow metabolism |
| **V29.5** | **5 min** | **~7 µs** | **180M** | **Enzyme kinetics!** |

At microsecond resolution we can capture:
- Enzyme catalytic cycles (~1-100 µs)
- Substrate binding/unbinding (~10-1000 µs)
- Conformational changes (~1-100 µs)
- Local metabolite diffusion (~100 µs)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

## 1. Network Definition (Same as V29.4)

In [ ]:
class GlycolysisNetwork:
    def __init__(self):
        self.metabolites = [
            'glc', 'g6p', 'f6p', 'fbp', 'gap', 'bpg', 'pep', 'pyr', 'lac',
            'atp', 'adp', 'nad', 'nadh', 'pi'
        ]
        self.n_met = len(self.metabolites)
        self.met_idx = {m: i for i, m in enumerate(self.metabolites)}
        
        # Reactions: (name, substrates, products, kcat, Km)
        self.reactions = [
            ('HK', [(0, 1), (9, 1)], [(1, 1), (10, 1)], 100.0, 0.1),
            ('PGI', [(1, 1)], [(2, 1)], 500.0, 0.5),
            ('PFK', [(2, 1), (9, 1)], [(3, 1), (10, 1)], 100.0, 0.1),
            ('ALDO', [(3, 1)], [(4, 2)], 50.0, 0.05),
            ('GAPDH', [(4, 1), (11, 1), (13, 1)], [(5, 1), (12, 1)], 100.0, 0.1),
            ('PGK', [(5, 1), (10, 1)], [(6, 1), (9, 1)], 500.0, 0.1),
            ('PK', [(6, 1), (10, 1)], [(7, 1), (9, 1)], 200.0, 0.1),
            ('LDH', [(7, 1), (12, 1)], [(8, 1), (11, 1)], 300.0, 0.2),
            ('ATPase', [(9, 1)], [(10, 1), (13, 1)], 100.0, 1.0),
        ]
        self.n_rxn = len(self.reactions)
        
        # Build stoichiometry
        self.S = np.zeros((self.n_met, self.n_rxn))
        self.kcat = np.zeros(self.n_rxn)
        self.Km = np.zeros(self.n_rxn)
        
        for j, (name, subs, prods, kcat, km) in enumerate(self.reactions):
            self.kcat[j] = kcat
            self.Km[j] = km
            for idx, coef in subs:
                self.S[idx, j] = -coef
            for idx, coef in prods:
                self.S[idx, j] = coef
        
        # Initial concentrations (mM)
        self.M0 = np.array([5.0, 0.1, 0.05, 0.02, 0.05, 0.001, 0.1, 0.1, 1.0,
                           3.0, 0.5, 1.0, 0.1, 5.0])

network = GlycolysisNetwork()
print(f"Network: {network.n_met} metabolites, {network.n_rxn} reactions")

## 2. Optimized Model for High-Frequency Simulation

In [ ]:
class FastMMModel(nn.Module):
    """Optimized for very high frequency simulation."""
    
    def __init__(self, network):
        super().__init__()
        
        self.n_met = network.n_met
        self.n_rxn = network.n_rxn
        
        self.register_buffer('S', torch.tensor(network.S, dtype=torch.float32))
        self.register_buffer('kcat', torch.tensor(network.kcat, dtype=torch.float32))
        self.register_buffer('Km', torch.tensor(network.Km, dtype=torch.float32))
        self.register_buffer('M0', torch.tensor(network.M0, dtype=torch.float32))
        
        # Pre-compute substrate indices as tensors for faster access
        # Create a mask: substrate_mask[j, i] = 1 if metabolite i is substrate of reaction j
        substrate_mask = torch.zeros(self.n_rxn, self.n_met)
        for j, (name, subs, prods, kcat, km) in enumerate(network.reactions):
            for idx, _ in subs:
                substrate_mask[j, idx] = 1.0
        self.register_buffer('substrate_mask', substrate_mask)
        
        # Count substrates per reaction for proper MM computation
        self.register_buffer('n_substrates', substrate_mask.sum(dim=1))
        
        # Small neural regulation (optional, can disable for pure MM)
        self.use_neural = False
        if self.use_neural:
            self.reg_net = nn.Sequential(
                nn.Linear(self.n_met, 32),
                nn.Tanh(),
                nn.Linear(32, self.n_rxn),
                nn.Tanh(),
            )
            self.reg_strength = nn.Parameter(torch.tensor(0.1))
    
    def forward(self, M, dt):
        """Vectorized forward pass for speed."""
        # Ensure positive
        M = torch.clamp(M, min=1e-8)
        
        # Compute saturation for all metabolites: sat = M / (Km + M)
        # Shape: (batch, n_met)
        Km_expanded = self.Km.unsqueeze(0).unsqueeze(-1)  # (1, n_rxn, 1)
        M_expanded = M.unsqueeze(1)  # (batch, 1, n_met)
        
        # For each reaction, we need product of saturations of its substrates
        # sat_all[batch, rxn, met] = M[batch, met] / (Km[rxn] + M[batch, met])
        sat_all = M_expanded / (self.Km.mean() + M_expanded)  # Simplified: use avg Km
        
        # Mask and compute product (using log-sum for numerical stability)
        # Where mask=0, we want sat=1 (no effect), so log(sat)=0
        log_sat = torch.log(sat_all + 1e-10)
        mask = self.substrate_mask.unsqueeze(0)  # (1, n_rxn, n_met)
        log_sat_masked = log_sat * mask
        log_product = log_sat_masked.sum(dim=-1)  # (batch, n_rxn)
        sat_product = torch.exp(torch.clamp(log_product, min=-20, max=20))
        
        # Flux = kcat * saturation_product
        v = self.kcat * sat_product
        
        # Rate of change
        dM_dt = torch.matmul(v, self.S.T)
        
        # Euler step
        M_next = M + dM_dt * dt
        M_next = torch.clamp(M_next, min=1e-8, max=100.0)
        
        return M_next, v
    
    @torch.no_grad()
    def simulate_fast(self, n_steps, dt, save_every, callback=None):
        """Optimized simulation loop."""
        M = self.M0.unsqueeze(0)
        
        n_save = n_steps // save_every + 1
        M_hist = torch.zeros(n_save, self.n_met, device=M.device)
        M_hist[0] = M[0]
        
        save_idx = 0
        
        for step in range(n_steps):
            M, _ = self.forward(M, dt)
            
            if (step + 1) % save_every == 0:
                save_idx += 1
                if save_idx < n_save:
                    M_hist[save_idx] = M[0]
                
                # Optional callback for progress
                if callback and (save_idx % 10000 == 0):
                    callback(step + 1, n_steps, M[0])
        
        return M_hist

model = FastMMModel(network).to(device)
print(f"Model ready on {device}")

# IMPORTANT: torch.compile with 'max-autotune' uses CUDA graphs which cause
# RuntimeError when tensors are reused in simulation loops.
# We disable compilation for stability.
USE_COMPILE = False
if USE_COMPILE:
    try:
        model = torch.compile(model, mode='reduce-overhead')
        print("Model compiled!")
    except Exception as e:
        print(f"Compilation not available: {e}")
else:
    print("Running without torch.compile (recommended for simulation loops)")

## 3. Benchmark to Determine Optimal Resolution

In [ ]:
# Benchmark to find actual steps/second on this hardware
print("Benchmarking simulation speed...")

# Warmup
M = model.M0.unsqueeze(0).to(device)
for _ in range(1000):
    M, _ = model.forward(M, 1e-6)

# Timed run
torch.cuda.synchronize() if device.type == 'cuda' else None
start = time.time()

test_steps = 100_000
M = model.M0.unsqueeze(0).to(device)
with torch.no_grad():
    for _ in range(test_steps):
        M, _ = model.forward(M, 1e-6)

torch.cuda.synchronize() if device.type == 'cuda' else None
elapsed = time.time() - start

steps_per_sec = test_steps / elapsed
print(f"\nBenchmark results:")
print(f"  {test_steps:,} steps in {elapsed:.2f} seconds")
print(f"  Speed: {steps_per_sec:,.0f} steps/second")

# Calculate optimal resolution for different time budgets
print(f"\n" + "="*60)
print("RESOLUTION OPTIONS (20 virtual minutes)")
print("="*60)

for budget_min in [0.5, 1, 2, 5, 10]:
    budget_sec = budget_min * 60
    max_steps = int(steps_per_sec * budget_sec)
    resolution_us = (20 * 60 * 1_000_000) / max_steps
    
    print(f"\n{budget_min} min real time:")
    print(f"  Steps: {max_steps:,}")
    print(f"  Resolution: {resolution_us:.1f} µs ({resolution_us/1000:.3f} ms)")
    
    if resolution_us < 10:
        print(f"  → Can capture enzyme catalysis!")
    elif resolution_us < 100:
        print(f"  → Can capture metabolite binding!")
    elif resolution_us < 1000:
        print(f"  → Can capture fast metabolism!")

## 4. Ultra-High Resolution Simulation (5 Real Minutes)

In [ ]:
# Configuration
REAL_TIME_BUDGET_MIN = 5  # Use 5 real minutes
VIRTUAL_DURATION_MIN = 20  # Simulate 20 virtual minutes
SAVE_EVERY = 1000  # Save every 1000 steps (for manageable output)

# Calculate steps based on benchmark
budget_sec = REAL_TIME_BUDGET_MIN * 60
n_steps = int(steps_per_sec * budget_sec * 0.9)  # 90% to leave margin
dt_min = VIRTUAL_DURATION_MIN / n_steps  # timestep in minutes
dt_us = dt_min * 60 * 1_000_000  # timestep in microseconds

print("="*65)
print("ULTRA-HIGH RESOLUTION SIMULATION")
print("="*65)
print(f"Real time budget: {REAL_TIME_BUDGET_MIN} minutes")
print(f"Virtual duration: {VIRTUAL_DURATION_MIN} minutes")
print(f"Total steps: {n_steps:,}")
print(f"Resolution: {dt_us:.2f} microseconds")
print(f"Saved points: {n_steps // SAVE_EVERY:,}")
print("="*65)

# Progress callback
start_time = time.time()
def progress_callback(step, total, M):
    elapsed = time.time() - start_time
    virtual_min = step * dt_min
    pct = 100 * step / total
    atp = M[network.met_idx['atp']].item()
    eta = (elapsed / step * total - elapsed) if step > 0 else 0
    print(f"  {pct:5.1f}% | Virtual: {virtual_min:5.1f} min | "
          f"Real: {elapsed:5.1f}s | ETA: {eta:5.0f}s | ATP: {atp:.2f} mM")

# Run simulation
print("\nStarting simulation...")
print("-" * 65)

M_history = model.simulate_fast(
    n_steps=n_steps,
    dt=dt_min,
    save_every=SAVE_EVERY,
    callback=progress_callback
)

total_time = time.time() - start_time
print("-" * 65)
print(f"\n✅ COMPLETE!")
print(f"   {VIRTUAL_DURATION_MIN} virtual minutes in {total_time:.1f} real seconds")
print(f"   Resolution achieved: {dt_us:.2f} microseconds")
print(f"   Data points: {len(M_history):,}")

In [ ]:
# Convert to numpy and create time array
M = M_history.cpu().numpy()
time_min = np.arange(len(M)) * SAVE_EVERY * dt_min
time_us = time_min * 60 * 1_000_000  # microseconds

print(f"Data shape: {M.shape}")
print(f"Time range: 0 to {time_min[-1]:.2f} minutes")
print(f"Time resolution of saved data: {SAVE_EVERY * dt_us:.1f} µs")

In [ ]:
# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Full trajectory - ATP/ADP
ax = axes[0, 0]
ax.plot(time_min, M[:, network.met_idx['atp']], 'b-', label='ATP', lw=0.3)
ax.plot(time_min, M[:, network.met_idx['adp']], 'r-', label='ADP', lw=0.3)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax.set_title(f'Energy Carriers (full 20 min, {dt_us:.1f}µs resolution)')
ax.legend()
ax.grid(True, alpha=0.3)

# Zoomed view - first 10 seconds
ax = axes[0, 1]
zoom_idx = time_min < 0.17  # ~10 seconds
ax.plot(time_min[zoom_idx] * 60, M[zoom_idx, network.met_idx['atp']], 'b-', lw=0.5)
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('ATP (mM)')
ax.set_title('ATP dynamics (first 10 seconds) - Microsecond detail!')
ax.grid(True, alpha=0.3)

# Metabolites
ax = axes[1, 0]
ax.plot(time_min, M[:, network.met_idx['glc']], label='Glucose', lw=0.3)
ax.plot(time_min, M[:, network.met_idx['pyr']], label='Pyruvate', lw=0.3)
ax.plot(time_min, M[:, network.met_idx['lac']], label='Lactate', lw=0.3)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax.set_title('Glycolysis substrates and products')
ax.legend()
ax.grid(True, alpha=0.3)

# Energy charge
atp = M[:, network.met_idx['atp']]
adp = M[:, network.met_idx['adp']]
ec = atp / (atp + adp + 0.01)

ax = axes[1, 1]
ax.plot(time_min, ec, 'purple', lw=0.3)
ax.axhline(0.85, color='g', ls='--', alpha=0.5)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Energy Charge')
ax.set_title('Cellular energy charge (ATP/(ATP+ADP))')
ax.set_ylim([0.5, 1.0])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v29_5_microsecond.png', dpi=150)
plt.show()

print(f"\nFinal state:")
print(f"  ATP: {atp[-1]:.3f} mM")
print(f"  ADP: {adp[-1]:.3f} mM") 
print(f"  Energy charge: {ec[-1]:.3f}")

In [ ]:
# Save results
output = {
    'metadata': {
        'model': 'Dark Manifold V29.5',
        'description': 'Microsecond resolution whole-cell simulation',
        'virtual_duration_min': VIRTUAL_DURATION_MIN,
        'real_time_sec': total_time,
        'resolution_us': dt_us,
        'total_steps': n_steps,
        'saved_points': len(M),
        'save_resolution_us': SAVE_EVERY * dt_us,
    },
    'time_min': time_min.tolist(),
    'metabolites': {name: M[:, i].tolist() for name, i in network.met_idx.items()},
    'energy_charge': ec.tolist(),
}

with open('whole_cell_v29_5_microsecond.json', 'w') as f:
    json.dump(output, f)

print(f"Saved to whole_cell_v29_5_microsecond.json")
print(f"File contains {len(M):,} timepoints at {SAVE_EVERY * dt_us:.1f}µs resolution")

## 5. Analysis: What Can We See at Microsecond Resolution?

In [ ]:
# Analyze high-frequency dynamics
print("="*60)
print("MICROSECOND DYNAMICS ANALYSIS")
print("="*60)

# ATP fluctuations
atp_diff = np.diff(atp)
print(f"\nATP dynamics:")
print(f"  Mean change per timestep: {np.mean(np.abs(atp_diff))*1000:.4f} µM")
print(f"  Max change: {np.max(np.abs(atp_diff))*1000:.4f} µM")
print(f"  Std of changes: {np.std(atp_diff)*1000:.4f} µM")

# Frequency analysis
from scipy import signal

# Compute power spectrum of ATP
fs = 1 / (SAVE_EVERY * dt_us * 1e-6)  # Sampling frequency in Hz
f, psd = signal.welch(atp - atp.mean(), fs, nperseg=min(len(atp)//4, 10000))

plt.figure(figsize=(10, 4))
plt.semilogy(f, psd)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density')
plt.title('ATP Concentration Fluctuation Spectrum')
plt.grid(True, alpha=0.3)
plt.xlim([0, min(fs/2, 1000)])  # Up to 1kHz or Nyquist
plt.show()

# Find dominant frequencies
peaks, _ = signal.find_peaks(psd, height=np.max(psd)/100)
if len(peaks) > 0:
    print(f"\nDominant frequencies in ATP dynamics:")
    for p in peaks[:5]:
        period_us = 1e6 / f[p] if f[p] > 0 else float('inf')
        print(f"  {f[p]:.1f} Hz (period: {period_us:.0f} µs)")